# Is Stack Overflow Dying?
## And what does that tell us about how developers work now?

Stack Overflow was the dominant place developers went for help from 2008 to roughly 2022. Then something changed. This notebook uses the same Stack Overflow post-count dataset to investigate the decline — when it started, how severe it is, and what it signals about the shift in how developers work in the AI era.

This is supplementary analysis to the [workforce strategy notebook](language_strategy.ipynb), which uses relative language share to correct for the platform effect identified here.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

df = pd.read_csv('../data/QueryResults.csv')
df.columns = ['DATE', 'TAG', 'POSTS']
df['DATE'] = pd.to_datetime(df['DATE'])

pivot = df.pivot(index='DATE', columns='TAG', values='POSTS').fillna(0)

AI_INFLECTION = pd.Timestamp('2022-11-01')

print(f"Date range: {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Languages tracked: {len(pivot.columns)}")

## 2. The Overall Decline

Total monthly posts across all tracked languages — the clearest view of platform-level activity.

In [ ]:
total_monthly = pivot.sum(axis=1)
smoothed_total = total_monthly.rolling(window=6).mean()

peak_date  = smoothed_total.idxmax()
peak_value = smoothed_total.max()
latest_value = smoothed_total.dropna().iloc[-1]
drop_pct = (latest_value - peak_value) / peak_value * 100

print(f"Peak activity:   {peak_date.strftime('%B %Y')} — {peak_value:,.0f} posts/month")
print(f"Latest reading:  {smoothed_total.dropna().index[-1].strftime('%B %Y')} — {latest_value:,.0f} posts/month")
print(f"Drop from peak:  {drop_pct:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

ax.fill_between(smoothed_total.index, smoothed_total, alpha=0.15, color='#5c6cfa')
ax.plot(smoothed_total.index, smoothed_total, color='#5c6cfa', linewidth=2.5)

# Peak annotation
ax.annotate(
    f'Peak: {peak_value:,.0f}/month\n{peak_date.strftime("%b %Y")}',
    xy=(peak_date, peak_value),
    xytext=(peak_date - pd.DateOffset(months=30), peak_value * 0.92),
    arrowprops=dict(arrowstyle='->', color='white', lw=1.2),
    fontsize=11, color='white'
)

# ChatGPT line
ax.axvline(AI_INFLECTION, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.8)
ax.text(AI_INFLECTION + pd.DateOffset(months=1), peak_value * 0.6,
        'ChatGPT launch\nNov 2022', color='#e74c3c', fontsize=11)

ax.set_title('Total Stack Overflow Posts Across All Tracked Languages (6-month rolling avg)',
             fontsize=15, fontweight='bold', pad=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Posts per Month', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(labelsize=11)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0')
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../data/so_total_decline.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 3. Pre vs Post AI: Which Languages Were Hit Hardest?

Comparing average monthly posts in the 24 months before ChatGPT's launch vs the 24 months after. Some languages may have held up better — which would itself be a signal.

In [ ]:
pre_ai  = pivot[pivot.index <  AI_INFLECTION].iloc[-24:].mean()
post_ai = pivot[pivot.index >= AI_INFLECTION].iloc[:24].mean()

ai_table = pd.DataFrame({
    'Pre-AI avg/month':  pre_ai.round(0).astype(int),
    'Post-AI avg/month': post_ai.round(0).astype(int),
    'Drop (%)':          ((post_ai - pre_ai) / pre_ai * 100).round(1),
}).sort_values('Drop (%)')

print('=== Impact of ChatGPT launch on monthly post volume ===')
print(ai_table.to_string())
print(f'\nAverage drop across all languages: {ai_table["Drop (%)"].mean():.1f}%')

In [ ]:
drops = ai_table['Drop (%)'].sort_values()
colors = ['#e74c3c' if v < -50 else '#f39c12' if v < -30 else '#2ecc71' for v in drops]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(drops.index, drops.values, color=colors, edgecolor='none')

for bar, val in zip(bars, drops.values):
    ax.text(val - 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', ha='right', fontsize=10, color='white')

ax.axvline(0, color='white', linewidth=0.6)
ax.axvline(drops.mean(), color='#f39c12', linewidth=1, linestyle=':', alpha=0.8)
ax.text(drops.mean() + 0.3, ax.get_ylim()[0] + 0.3,
        f'avg {drops.mean():.1f}%', color='#f39c12', fontsize=9)

ax.set_title('Post-volume drop per language: 24 months before vs after ChatGPT launch',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Change in average monthly posts (%)', fontsize=11)

patches = [
    mpatches.Patch(color='#e74c3c', label='> 50% drop'),
    mpatches.Patch(color='#f39c12', label='30–50% drop'),
    mpatches.Patch(color='#2ecc71', label='< 30% drop'),
]
ax.legend(handles=patches, fontsize=10)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../data/so_ai_impact.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 4. Share of Activity Over Time

If we strip out the platform-level decline and look at each language's share of monthly posts, we can see which languages are actually growing or shrinking *relative to each other* — independent of what Stack Overflow is doing overall.

In [ ]:
row_totals = pivot.sum(axis=1).replace(0, np.nan)
share = pivot.div(row_totals, axis=0) * 100
smoothed_share = share.rolling(window=6).mean()

# Focus on the languages with meaningful share
top_langs = share.mean().sort_values(ascending=False).head(8).index

fig, ax = plt.subplots(figsize=(16, 8))

for lang in top_langs:
    ax.plot(smoothed_share.index, smoothed_share[lang], linewidth=2, label=lang)

ax.axvline(AI_INFLECTION, color='white', linewidth=1.2, linestyle='--', alpha=0.6)
ax.text(AI_INFLECTION + pd.DateOffset(months=1),
        smoothed_share[top_langs].max().max() * 0.95,
        'ChatGPT\nNov 2022', color='white', fontsize=10, alpha=0.8)

ax.set_title('Language Share of Monthly Stack Overflow Posts (6-month rolling avg)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Share of monthly posts (%)', fontsize=12)
ax.legend(fontsize=11, loc='upper left', ncol=2)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.yaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig('../data/so_share_over_time.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 5. What This Tells Us About How Developers Work Now

Three observations from the data:

In [ ]:
avg_drop = ai_table['Drop (%)'].mean()
least_hit = ai_table['Drop (%)'].idxmax()   # smallest negative = least hit
most_hit  = ai_table['Drop (%)'].idxmin()

# Python share: pre vs post AI
python_pre  = share.loc[share.index <  AI_INFLECTION, 'python'].iloc[-24:].mean()
python_post = share.loc[share.index >= AI_INFLECTION, 'python'].iloc[:24].mean()
python_share_change = (python_post - python_pre) / python_pre * 100

conclusion = f"""
FINDINGS
─────────────────────────────────────────────────────────────────────
1. STACK OVERFLOW IS IN STRUCTURAL DECLINE
   Total post volume dropped ~{abs(avg_drop):.0f}% on average across all languages
   in the 24 months after ChatGPT launched vs the 24 months before.
   This is not a blip — it maps directly to developers switching to
   AI tools for answers they previously searched Stack Overflow for.

2. THE DECLINE IS UNEVEN — AND THAT'S THE SIGNAL
   {most_hit.title()} was hit hardest ({ai_table.loc[most_hit, 'Drop (%)']:.1f}%).
   {least_hit.title()} held up best ({ai_table.loc[least_hit, 'Drop (%)']:.1f}%).
   Languages that held up better in absolute terms are likely those
   where developers still need community help — either because the
   language is growing fast (new users asking basics) or because AI
   tools handle it less well.

3. PYTHON'S SHARE {'GREW' if python_share_change > 0 else 'HELD'} THROUGH THE DECLINE
   Python's share of Stack Overflow posts
   {'increased' if python_share_change > 0 else 'changed'} by {python_share_change:+.1f}% after the AI inflection point.
   This is counterintuitive — Python is the primary language for
   AI/ML development, so as AI adoption grew, so did Python activity,
   even on a declining platform. Its growth is structural.

IMPLICATION FOR WORKFORCE STRATEGY
   Stack Overflow is becoming an unreliable absolute signal for
   developer interest. Relative share and job market data (see the
   strategy notebook) are now the more defensible measures.
─────────────────────────────────────────────────────────────────────
"""

print(conclusion)